# NYC Building Permits ETL Pipeline
## Load Phase

Cleaned permit data is loaded from local Parquet storage into 
a Snowflake data warehouse, establishing the final analytical 
layer for BI and reporting use cases.

**Input:** Cleaned Parquet file from Transform phase  
**Destination:** Snowflake — NYC_PERMITS database  
**Schema:** Public

#### Import Libraries
Libraries for distributed data processing, Snowflake 
connectivity, and environment variable management.

In [1]:
# Import libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import snowflake.connector
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

print("Libraries loaded successfully")

Libraries loaded successfully


#### Initialize Spark Session
A local SparkSession is initialized to read the cleaned 
Parquet file produced in the Transform phase.

In [2]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("NYC Permits Load") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
print("Spark session initialized successfully")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/22 15:59:35 WARN Utils: Your hostname, Catalinas-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.16 instead (on interface en0)
26/03/22 15:59:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/22 15:59:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/22 15:59:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.1
Spark session initialized successfully


#### Connect to Snowflake
Connection credentials are loaded from environment variables 
and a session is established with the Snowflake data warehouse.

In [3]:
conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
    schema=os.getenv("SNOWFLAKE_SCHEMA")
)

cursor = conn.cursor()
print("Snowflake connection established successfully")

Snowflake connection established successfully


#### Create Database and Table
The target database, schema, and table are created in Snowflake 
if they do not already exist, enforcing the schema definition 
for all incoming permit records.

In [7]:
# Create database and table
cursor.execute("CREATE DATABASE IF NOT EXISTS NYC_PERMITS")
cursor.execute("USE DATABASE NYC_PERMITS")
cursor.execute("USE SCHEMA PUBLIC")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS BUILDING_PERMITS (
        borough STRING,
        zip_code STRING,
        block STRING,
        lot STRING,
        job_type STRING,
        permit_type STRING,
        permit_status STRING,
        filing_date TIMESTAMP,
        issuance_date TIMESTAMP,
        expiration_date TIMESTAMP,
        work_type STRING,
        residential STRING,
        bldg_type STRING,
        owner_s_business_name STRING,
        owner_s_business_type STRING,
        permittee_s_business_name STRING,
        permittee_s_license_type STRING,
        gis_latitude DOUBLE,
        gis_longitude DOUBLE,
        gis_nta_name STRING,
        permit_year INT,
        permit_month INT,
        days_to_expiration INT,
        is_residential BOOLEAN
    )
""")

print("Database and table created successfully")

Database and table created successfully


#### Read Cleaned Parquet File
The cleaned Parquet dataset produced in the Transform phase 
is loaded into a Spark DataFrame for final preparation 
prior to loading into Snowflake.

In [8]:
# Read cleaned parquet file
df = spark.read.parquet(
    "/Users/catalinavalencia/Documents/ETL-ELT Project/data/cleaned_permits.parquet"
)

print(f"Rows loaded: {df.count()}")
print(f"Columns: {len(df.columns)}")

Rows loaded: 50000
Columns: 24


#### Write Data to Snowflake
The cleaned dataset is converted and loaded into the Snowflake 
BUILDING_PERMITS table in batches, completing the ETL pipeline.

In [12]:
from snowflake.connector.pandas_tools import write_pandas

# Convert Spark DataFrame to Pandas
df_pandas = df.toPandas()

# Convert timestamp columns to strings
date_cols = ["filing_date", "issuance_date", "expiration_date"]
for col in date_cols:
    df_pandas[col] = df_pandas[col].astype(str).replace("NaT", None)

# Replace all NaN with None
df_pandas = df_pandas.where(pd.notnull(df_pandas), None)

# Uppercase column names for Snowflake
df_pandas.columns = [c.upper() for c in df_pandas.columns]

# Write to Snowflake
success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=df_pandas,
    table_name="BUILDING_PERMITS",
    database="NYC_PERMITS",
    schema="PUBLIC"
)

print(f"Success: {success}")
print(f"Chunks: {nchunks}")
print(f"Total rows loaded: {nrows}")

Success: True
Chunks: 1
Total rows loaded: 50000


#### Verify Data in Snowflake
A validation query confirms successful data delivery by 
checking row counts and previewing loaded records.

In [13]:
# Verify data in Snowflake
cursor.execute("SELECT COUNT(*) FROM BUILDING_PERMITS")
count = cursor.fetchone()[0]
print(f"Rows in Snowflake: {count}")

cursor.execute("SELECT BOROUGH, PERMIT_TYPE, FILING_DATE, WORK_TYPE FROM BUILDING_PERMITS LIMIT 5")
rows = cursor.fetchall()
print("\nSample records:")
for row in rows:
    print(row)

# Close connection
cursor.close()
conn.close()
print("\nSnowflake connection closed.")

Rows in Snowflake: 50000

Sample records:
('BROOKLYN', 'NB', datetime.datetime(2026, 3, 20, 0, 0), 'UNKNOWN')
('BROOKLYN', 'AL', datetime.datetime(2026, 3, 20, 0, 0), 'UNKNOWN')
('BROOKLYN', 'PL', datetime.datetime(2026, 3, 20, 0, 0), 'PL')
('BROOKLYN', 'PL', datetime.datetime(2026, 3, 20, 0, 0), 'PL')
('BROOKLYN', 'EW', datetime.datetime(2026, 3, 20, 0, 0), 'SP')

Snowflake connection closed.
